# Retrieval Evaluation Runner

This notebook:
- loads [prompt.md](/d:/Artificial%20Intelligence/School%20Courses/Software%20Engineering%20Project/IUCN_Reviewer/llm_rag/evaluation/prompt.md)
- loads a retrieval-evaluation JSON payload such as `Q1_retrieval_eval.json`
- replaces `{...JSON loaded...}` in the prompt with the payload text
- sends the full prompt to an Ollama endpoint
- parses the model response into JSON for inspection


In [1]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

import requests
from ollama import Client

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
if not (REPO_ROOT / "llm_rag").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent.parent

EVAL_DIR = REPO_ROOT / "llm_rag" / "evaluation"
PROMPT_PATH = EVAL_DIR / "prompt.md"
PAYLOAD_PATH = EVAL_DIR / "Q1_retrieval_eval.json"

os.environ["OLLAMA_API_KEY"] = "718f6f449c2e4bfe9cde7a0f3651360d.aPXqUP_YlcwFWmrFAuBMszFB"
client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
)

In [2]:
prompt_template = PROMPT_PATH.read_text(encoding="utf-8")
payload_text = PAYLOAD_PATH.read_text(encoding="utf-8")

# Use the raw file text for prompt injection so the workflow still works even if the
# payload file is not perfectly valid JSON yet.
filled_prompt = prompt_template.replace("{...JSON loaded...}", payload_text)

try:
    payload_json = json.loads(payload_text)
    payload_is_valid_json = True
except json.JSONDecodeError as exc:
    payload_json = None
    payload_is_valid_json = False
    print(f"Payload is not valid JSON yet: {exc}")

print(f"Prompt chars:  {len(prompt_template)}")
print(f"Payload chars: {len(payload_text)}")
print(f"Final prompt chars: {len(filled_prompt)}")
print(f"Payload valid JSON: {payload_is_valid_json}")


Prompt chars:  3691
Payload chars: 3770
Final prompt chars: 7442
Payload valid JSON: True


In [3]:
# Preview the tail of the prompt to confirm that the payload was inserted.

print(filled_prompt[-3000:])


re are threats in the area, a listing of Critically Endangered (Possibly Extinct) or Extinct may be appropriate (see section 11 for guidance on how to make this determination). 8.2 Example of applying criterion D"
    },
    {
      "block type": "text",
      "source": "RedListGuidelines.pdf",
      "page": 11,
      "section": "Figure 2.1. Structure of the IUCN Red List Categories",
      "text": ". A taxon is presumed Extinct in the Wild when exhaustive surveys in known and/or expected habitat, at appropriate times (diurnal, seasonal, annual), throughout its historic range have failed to record an individual. Surveys should be over a time frame appropriate to the taxon's life cycle and life form. CRITICALLY ENDANGERED (CR) A taxon is Critically Endangered when the best available evidence indicates that it meets any of the criteria A to E for Critically Endangered, and it is therefore considered to be facing an extremely high risk of extinction in the wild. ENDANGERED (EN) A taxon is

In [5]:
messages = [
  {
    'role': 'system',
    'content': filled_prompt
  },
]

response = client.chat('gpt-oss:120b', messages=messages)

output_path = EVAL_DIR / "Q1_retrieval_eval_result.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(response['message']['content'], f, ensure_ascii=False, indent=2)